In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into kernel variables (kept in memory)
metric_df = pd.read_csv('metric.csv', low_memory=False)
trace_df = pd.read_csv('trace.csv', low_memory=False)
log_df = pd.read_csv('log.csv', low_memory=False)
error_logs_df = pd.read_csv('error_logs.csv', low_memory=False)

# Convert timestamp to timezone-aware datetimes (UTC->Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_logs_df):
    # ensure timestamp exists and is numeric
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper to build compact summary
def compact_summary(df, name, id_col='cmdb_id'):
    rows = df.shape[0]
    min_ts = df['dt'].min() if 'dt' in df.columns else pd.NaT
    max_ts = df['dt'].max() if 'dt' in df.columns else pd.NaT
    top_cmdb = df[id_col].value_counts().head(20).reset_index()
    top_cmdb.columns = [id_col, 'count']
    return pd.DataFrame({
        'file': [name],
        'rows': [rows],
        'min_ts': [min_ts],
        'max_ts': [max_ts]
    }), top_cmdb

# metric summaries
metric_summary_df, metric_top_cmdb = compact_summary(metric_df, 'metric.csv')
metric_kpi_counts = metric_df['kpi_name'].value_counts().head(20).reset_index()
metric_kpi_counts.columns = ['kpi_name', 'count']

# trace summaries
trace_summary_df, trace_top_cmdb = compact_summary(trace_df, 'trace.csv')
trace_name_counts = trace_df['trace_name'].value_counts().head(20).reset_index()
trace_name_counts.columns = ['trace_name', 'count']

# log summaries
log_summary_df, log_top_cmdb = compact_summary(log_df, 'log.csv')
log_name_counts = log_df['log_name'].value_counts().head(20).reset_index()
log_name_counts.columns = ['log_name', 'count']

# error_logs summaries (message samples)
error_summary_df, error_top_cmdb = compact_summary(error_logs_df, 'error_logs.csv')
if 'message' in error_logs_df.columns:
    error_message_counts = error_logs_df['message'].value_counts().head(20).reset_index()
    error_message_counts.columns = ['message_sample', 'count']
else:
    error_message_counts = pd.DataFrame(columns=['message_sample','count'])

# Display compact summaries (limited rows and columns)
metric_summary_df, metric_top_cmdb, metric_kpi_counts, \
trace_summary_df, trace_top_cmdb, trace_name_counts, \
log_summary_df, log_top_cmdb, log_name_counts, \
error_summary_df, error_top_cmdb, error_message_counts

```
Out[1]:
```
```python
# Summarize previous execution results in plain English (compact)
summary = (
    "Summary of loaded telemetry files (compact):\n\n"
    "metric.csv:\n"
    "- Rows: 144,252; Time range: 2022-03-20 15:00 to 2022-03-20 15:29 (UTC+8)\n"
    "- Top cmdb_id by rows (top 5): adservice(10230), adservice2(9990), frontend-0(4800), frontend2-0(4800), frontend-1(4680)\n"
    "- Top kpi_name samples (top 5): container.node-5.container_network_transmit_MB...(1050), container.node-5.container_network_transmit_errors...(1050), container.node-5.container_network_transmit_packets...(1050), container.node-5.container_network_receive_packets...(1050), container.node-5.container_network_receive_packets...(1050)\n\n"
    "trace.csv:\n"
    "- Rows: 36,520; Time range: 2022-03-20 15:00 to 2022-03-20 15:29 (UTC+8)\n"
    "- Top cmdb_id by rows (top 5): frontend-2(2752), frontend-1(2736), frontend-0(2728), checkoutservice-0(2700), checkoutservice-1(2680)\n"
    "- Top trace_name samples (top 5): trace.from_frontend-2.row_count(628), trace.from_frontend-2.error_rate(628), trace.from_frontend-2.duration_p95(628), trace.from_frontend-2.duration_mean(628), trace.from_frontend-1.row_count(624)\n\n"
    "log.csv:\n"
    "- Rows: 1,814; Time range: 2022-03-20 15:00 to 2022-03-20 15:29 (UTC+8)\n"
    "- Top cmdb_id by rows (top examples): adservice-0(60), adservice-1(60), adservice-2(60), cartservice-0(60), cartservice-1(60)\n"
    "- Top log_name values: log.error_count(907), log.row_count(907)\n\n"
    "error_logs.csv:\n"
    "- Rows: 1,925; Time range: 2022-03-20 15:00 to 2022-03-20 15:29:59 (UTC+8)\n"
    "- Top cmdb_id by rows: frontend-2(688), frontend-0(688), frontend-1(549)\n"
    "- Top message sample: 'severity: warning, message: failed to retrieve...' (appears 1,925 times)\n\n"
    "Notes:\n"
    "- Data is concentrated in the 15:00–15:29 UTC+8 window across files.\n"
    "- Many metric KPIs reference container.node-5 network and memory-related metrics.\n"
    "- Trace and error logs are dominated by frontend pods (frontend-0/1/2), suggesting frontend-related activity/errors during this period.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file    rows                    min_ts                    max_ts
0  metric.csv  144252 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00,                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                 frontend-0   4800
3                frontend2-0   4800
4                 frontend-1   4680
5                 frontend-2   4380
6          checkoutservice-0   3780
7          checkoutservice-1   3540
8          checkoutservice-2   3540
9    productcatalogservice-0   3480
10   productcatalogservice-2   3480
11   productcatalogservice-1   3360
12        checkoutservice2-0   3300
13  productcatalogservice2-0   2880
14             cartservice-1   2760
15             cartservice-0   2640
16            cartservice2-0   2640
17              adservice2-0   2580
18               adservice-0   2580
19             cartservice-2   2520,                                              kpi_name  count
0   container.node-5.container_network_transmit_MB...   1050
1   container.node-5.container_network_transmit_er...   1050
2   container.node-5.container_network_transmit_pa...   1050
3   container.node-5.container_network_receive_pac...   1050
4   container.node-5.container_network_receive_pac...   1050
5   container.node-5.container_network_transmit_pa...   1050
6   container.node-5.container_network_receive_MB....   1050
7   container.node-5.container_network_receive_err...   1050
8              container.node-5.container_threads_max    990
9   container.node-5.container_ulimits_soft.max_op...    990
10  container.node-5.container_spec_memory_reserva...    990
11    container.node-5.container_spec_memory_limit_MB    990
12     container.node-5.container_tasks_state.running    990
13         container.node-5.container_spec_cpu_period    990
14      container.node-5.container_memory_mapped_file    990
15  container.node-5.container_memory_failures.con...    990
16               container.node-5.container_last_seen    990
17          container.node-5.container_spec_cpu_quota    990
18            container.node-5.container_memory_cache    990
19          container.node-5.container_memory_failcnt    990,         file   rows                    min_ts                    max_ts
0  trace.csv  36520 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00,                     cmdb_id  count
0                frontend-2   2752
1                frontend-1   2736
2                frontend-0   2728
3         checkoutservice-0   2700
4         checkoutservice-1   2680
5         checkoutservice-2   2680
6   productcatalogservice-1   1200
7   productcatalogservice-0   1196
8   productcatalogservice-2   1192
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13  recommendationservice-1    840
14            cartservice-2    836
15            cartservice-1    836
16            cartservice-0    836
17        currencyservice-0    720
18        currencyservice-1    720
19        currencyservice-2    720,                                     trace_name  count
0              trace.from_frontend-2.row_count    628
1             trace.from_frontend-2.error_rate    628
2           trace.from_frontend-2.duration_p95    628
3          trace.from_frontend-2.duration_mean    628
4              trace.from_frontend-1.row_count    624
5             trace.from_frontend-1.error_rate    624
6           trace.from_frontend-1.duration_p95    624
7          trace.from_frontend-1.duration_mean    624
8          trace.from_frontend-0.duration_mean    622
9           trace.from_frontend-0.duration_p95    622
10            trace.from_frontend-0.error_rate    622
11             trace.from_frontend-0.row_count    622
12      trace.from_checkoutservice-0.row_count    530
13     trace.from_checkoutservice-0.error_rate    530
14   trace.from_checkoutservice-0.duration_p95    530
15  trace.from_checkoutservice-0.duration_mean    530
16      trace.from_checkoutservice-2.row_count    527
17     trace.from_checkoutservice-2.error_rate    527
18   trace.from_checkoutservice-2.duration_p95    527
19  trace.from_checkoutservice-2.duration_mean    527,       file  rows                    min_ts                    max_ts
0  log.csv  1814 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00,               cmdb_id  count
0         adservice-0     60
1         adservice-1     60
2         adservice-2     60
3       cartservice-0     60
4       cartservice-1     60
5       cartservice-2     60
6   checkoutservice-0     60
7   checkoutservice-1     60
8   checkoutservice-2     60
9   currencyservice-0     60
10  currencyservice-1     60
11  currencyservice-2     60
12     emailservice-0     60
13     emailservice-1     60
14     emailservice-2     60
15         frontend-0     60
16         frontend-1     60
17         frontend-2     60
18   paymentservice-0     60
19   paymentservice-1     60,           log_name  count
0  log.error_count    907
1    log.row_count    907,              file  rows                    min_ts                    max_ts
0  error_logs.csv  1925 2022-03-20 15:00:00+08:00 2022-03-20 15:29:59+08:00,       cmdb_id  count
0  frontend-2    688
1  frontend-0    688
2  frontend-1    549,                                       message_sample  count
0  severity: warning, message: failed to retrieve...   1925)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# ensure dt exists and is timezone-aware UTC+8
if 'dt' not in metric_df.columns:
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce')
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware
    if metric_df['dt'].dt.tz is None:
        metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series
thresholds = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p5 = lambda x: x.quantile(0.05)
).reset_index()

# 2) Filter to incident window 2022-03-20 15:00–15:30 (UTC+8)
start = pd.Timestamp('2022-03-20 15:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 15:30:00', tz=tz)  # exclusive
window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] < end)].copy()

# merge thresholds into windowed data
window_merged = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# mark anomalies: value >= p95 OR value <= p5
window_merged['anomaly'] = (window_merged['value'] >= window_merged['p95']) | (window_merged['value'] <= window_merged['p5'])

# 3) Aggregate per (cmdb_id, kpi_name)
totals = window_merged.groupby(['cmdb_id','kpi_name']).size().reset_index(name='total_points_in_window')

anomalies = window_merged[window_merged['anomaly']].copy()
anomaly_agg = anomalies.groupby(['cmdb_id','kpi_name']).agg(
    anomaly_count = ('anomaly','size'),
    earliest_anomaly_timestamp = ('dt','min'),
    latest_anomaly_timestamp = ('dt','max')
).reset_index()

# combine totals, anomaly aggregates, and thresholds (p95/p5)
result = totals.merge(anomaly_agg, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# fill zeros for pairs without anomalies
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)

# sort by anomaly_count desc and keep top 20
top20_pairs = result.sort_values('anomaly_count', ascending=False).head(20)

# 4) counts
distinct_pairs_with_anomaly = int((result['anomaly_count'] >= 1).sum())
total_anomaly_rows = int(anomalies.shape[0])

# Keep outputs compact: show top20 selected columns
top20_display = top20_pairs[[
    'cmdb_id','kpi_name','total_points_in_window','anomaly_count',
    'earliest_anomaly_timestamp','latest_anomaly_timestamp','p95','p5'
]].reset_index(drop=True)

# Display compact results (variables kept in kernel for later use)
thresholds, window_merged.shape, top20_display, distinct_pairs_with_anomaly, total_anomaly_rows

```
Out[2]:
```
```python
# Compact plain-English summary string based on previous results in kernel
summary = (
    f"Computed global P95/P5 for {len(thresholds)} unique (cmdb_id, kpi_name) pairs.\n"
    f"Incident window (2022-03-20 15:00–15:30 UTC+8) contains {window_merged.shape[0]} metric points.\n"
    f"Total anomaly points in that window (value >= P95 or <= P5): {total_anomaly_rows}.\n"
    f"Distinct (cmdb_id, kpi_name) pairs with >=1 anomaly: {distinct_pairs_with_anomaly}.\n\n"
    f"Top-20 pairs are dominated by shippingservice2-0 container/node KPIs; several top pairs had "
    f"all points in-window flagged as anomalies (e.g., 30/30). Earliest anomaly in-window: "
    f"{anomalies['dt'].min()}, latest anomaly in-window: {anomalies['dt'].max()}.\n\n"
    "Quick observation: many KPIs show zero or constant thresholds (P95 == P5 or P95/P5 == 0), "
    "so entire per-minute series for those KPIs in the window are marked anomalous. This may reflect "
    "low-variance or quantized metrics rather than transient spikes; consider inspecting raw KPI semantics "
    "before acting on these alerts."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name           p95            p5
0              adservice                                     app.grpc.count    248.300000    210.450000
1              adservice                                       app.grpc.mrt      0.002387      0.002244
2              adservice                                        app.grpc.rr    100.000000    100.000000
3              adservice                                        app.grpc.sr     57.950512     52.607562
4              adservice                                     app.http.count     24.000000     24.000000
...                  ...                                                ...           ...           ...
4813  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...     16.550000      8.500000
4814  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  45820.000000  28831.250000
4815  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      0.001339      0.000000
4816  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     15.775000      9.000000
4817  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....   1569.612500    967.637500

[4818 rows x 4 columns], (144252, 8),                cmdb_id                                           kpi_name  total_points_in_window  anomaly_count earliest_anomaly_timestamp  latest_anomaly_timestamp         p95          p5
0   shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
1   shippingservice2-0  container.node-5.container_ulimits_soft.max_op...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00     65536.0     65536.0
2   shippingservice2-0     container.node-5.container_fs_writes./dev/vda1                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
3   shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
4   shippingservice2-0  container.node-5.container_fs_writes_merged./d...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
5   shippingservice2-0            container.node-5.container_memory_cache                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00  18464768.0  18464768.0
6   shippingservice2-0          container.node-5.container_memory_failcnt                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00   2397029.0   2397029.0
7   shippingservice2-0  container.node-5.container_memory_failures.con...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
8   shippingservice2-0      container.node-5.container_fs_reads./dev/vda1                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
9   shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
10  shippingservice2-0  container.node-5.container_fs_reads_merged./de...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
11  shippingservice2-0  container.node-5.container_fs_sector_reads./de...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
12  shippingservice2-0  container.node-5.container_fs_sector_writes./d...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
13  shippingservice2-0  container.node-5.container_fs_write_seconds./d...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
14  shippingservice2-0      container.node-5.container_fs_writes./dev/vda                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
15  shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
16  shippingservice2-0  container.node-5.container_fs_inodes_free./dev...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
17  shippingservice2-0  container.node-5.container_fs_io_current./dev/...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
18  shippingservice2-0  container.node-5.container_fs_io_time_seconds....                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0
19  shippingservice2-0  container.node-5.container_fs_io_time_weighted...                      30             30  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00         0.0         0.0, 4818, 101374)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: anomalies, window_merged, totals
# Ensure dt is timezone-aware
if 'dt' not in anomalies.columns:
    anomalies['timestamp'] = pd.to_numeric(anomalies['timestamp'], errors='coerce')
    anomalies['dt'] = pd.to_datetime(anomalies['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if anomalies['dt'].dt.tz is None:
        anomalies['dt'] = pd.to_datetime(anomalies['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Aggregate anomalies by cmdb_id
agg_by_comp = anomalies.groupby('cmdb_id').agg(
    total_anomaly_count = ('anomaly', 'size'),
    num_distinct_kpi_pairs_with_anomaly = ('kpi_name', 'nunique'),
    earliest_anomaly_timestamp = ('dt', 'min'),
    latest_anomaly_timestamp = ('dt', 'max')
).reset_index()

# total_points_in_window per cmdb_id (sum across its KPIs) from the 'totals' df
# totals has columns ['cmdb_id','kpi_name','total_points_in_window']
total_points_per_comp = totals.groupby('cmdb_id')['total_points_in_window'].sum().reset_index()

# merge
agg_by_comp = agg_by_comp.merge(total_points_per_comp, on='cmdb_id', how='left')

# fill missing total_points_in_window with 0
agg_by_comp['total_points_in_window'] = agg_by_comp['total_points_in_window'].fillna(0).astype(int)

# sort by total_anomaly_count desc and keep top 20
top_components = agg_by_comp.sort_values('total_anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Select requested columns (compact)
top_components = top_components[[
    'cmdb_id','total_anomaly_count','num_distinct_kpi_pairs_with_anomaly',
    'total_points_in_window','earliest_anomaly_timestamp','latest_anomaly_timestamp'
]]

# 2) For top 5 components, produce minute-level anomaly counts within the incident window
top5_list = top_components['cmdb_id'].head(5).tolist()

# compute minute-level anomaly counts for anomalies (group by component and dt)
minute_counts = (
    anomalies[anomalies['cmdb_id'].isin(top5_list)]
    .groupby(['cmdb_id','dt'])
    .size()
    .reset_index(name='anomaly_count')
    .sort_values(['cmdb_id','dt'])
)

# Cap printed rows per component to at most 20: take head(20) per cmdb_id
minute_counts_limited = minute_counts.groupby('cmdb_id').head(20).reset_index(drop=True)

# Keep outputs compact
top_components, minute_counts_limited.shape, minute_counts_limited.head(100)

```
Out[3]:
```
```python
# Compact plain-English summary using existing kernel variables (top_components, minute_counts_limited, distinct_pairs_with_anomaly, total_anomaly_rows)
tz_str = "UTC+8"

top5 = top_components.head(5)

# Build per-component minute-level stats from minute_counts_limited
per_comp_stats = []
for cm in top5['cmdb_id'].tolist():
    comp_minutes = minute_counts_limited[minute_counts_limited['cmdb_id']==cm]
    minutes_shown = int(comp_minutes.shape[0])
    if minutes_shown > 0:
        min_count = int(comp_minutes['anomaly_count'].min())
        max_count = int(comp_minutes['anomaly_count'].max())
    else:
        min_count = max_count = 0
    per_comp_stats.append((cm, minutes_shown, min_count, max_count))

summary_lines = []
summary_lines.append(f"Anomaly aggregation summary (incident window 2022-03-20 15:00–15:30 {tz_str}):")
summary_lines.append(f"- Total anomaly rows across metrics: {int(total_anomaly_rows)}")
summary_lines.append(f"- Distinct (cmdb_id, kpi_name) pairs with >=1 anomaly: {int(distinct_pairs_with_anomaly)}")
summary_lines.append("")
summary_lines.append("Top 5 components by anomaly count (cmdb_id | total_anomaly_count | distinct_kpi_pairs_with_anomaly | total_points_in_window):")

for _, r in top5.iterrows():
    summary_lines.append(
        f"  - {r['cmdb_id']} | {int(r['total_anomaly_count'])} | "
        f"{int(r['num_distinct_kpi_pairs_with_anomaly'])} | {int(r['total_points_in_window'])}"
    )

summary_lines.append("")
summary_lines.append("Minute-level anomaly count overview for these top 5 components (rows shown per component capped at 20):")
for cm, minutes_shown, min_c, max_c in per_comp_stats:
    summary_lines.append(
        f"  - {cm}: minutes_shown={minutes_shown}, anomaly_count_range_per_minute={min_c}–{max_c}"
    )

summary_lines.append("")
summary_lines.append("Quick observations:")
summary_lines.append("  - adservice and adservice2 have the highest anomaly totals (~9k each), covering many KPIs.")
summary_lines.append("  - Frontend pods (frontend2-0, frontend-0, frontend-1) are next, each with ~2.7k–3.0k anomalies.")
summary_lines.append("  - Minute-level counts indicate sustained, high-frequency anomalies (e.g., adservice minutes ~300 anomalies/min; frontends ~90–100 anomalies/min).")
summary_lines.append("  - Many anomalies arise from many KPIs per component; inspect KPI semantics (some have P95==P5 or constant values) before acting.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  total_anomaly_count  num_distinct_kpi_pairs_with_anomaly  total_points_in_window earliest_anomaly_timestamp  latest_anomaly_timestamp
0                  adservice                 9195                                  341                   10230  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
1                 adservice2                 8912                                  333                    9990  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
2                frontend2-0                 3023                                  160                    4800  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
3                 frontend-0                 2915                                  160                    4800  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
4                 frontend-1                 2797                                  156                    4680  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
5                 frontend-2                 2511                                  146                    4380  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
6    productcatalogservice-0                 2347                                  116                    3480  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
7    productcatalogservice-2                 2333                                  116                    3480  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
8    productcatalogservice-1                 2244                                  112                    3360  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
9          checkoutservice-0                 2209                                  126                    3780  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
10         checkoutservice-1                 2064                                  118                    3540  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
11               adservice-0                 2019                                   86                    2580  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
12         checkoutservice-2                 2019                                  118                    3540  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
13             cartservice-1                 1970                                   92                    2760  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
14  productcatalogservice2-0                 1966                                   96                    2880  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
15              adservice2-0                 1924                                   86                    2580  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
16            cartservice2-0                 1906                                   88                    2640  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
17             cartservice-0                 1848                                   88                    2640  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
18               adservice-1                 1840                                   82                    2460  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
19   recommendationservice-1                 1818                                   80                    2400  2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00, (100, 3),         cmdb_id                        dt  anomaly_count
0     adservice 2022-03-20 15:00:00+08:00            327
1     adservice 2022-03-20 15:01:00+08:00            326
2     adservice 2022-03-20 15:02:00+08:00            305
3     adservice 2022-03-20 15:03:00+08:00            308
4     adservice 2022-03-20 15:04:00+08:00            305
..          ...                       ...            ...
95  frontend2-0 2022-03-20 15:15:00+08:00             97
96  frontend2-0 2022-03-20 15:16:00+08:00             98
97  frontend2-0 2022-03-20 15:17:00+08:00             92
98  frontend2-0 2022-03-20 15:18:00+08:00             96
99  frontend2-0 2022-03-20 15:19:00+08:00             99

[100 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
large_inf = 1e9  # represent +Inf for breach_ratio

# Reuse anomalies and thresholds from kernel (anomalies contains rows flagged as anomaly within window)
df = anomalies.copy()

# Ensure dt exists and is tz-aware
if 'dt' not in df.columns:
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df['dt'].dt.tz is None:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

df = df.sort_values(['cmdb_id','kpi_name','dt']).reset_index(drop=True)

# Assign consecutive-run sequence within each (cmdb_id, kpi_name)
def assign_run_ids(g):
    g = g.sort_values('dt').copy()
    dt_diff = g['dt'].diff()
    new_run = (dt_diff != pd.Timedelta(minutes=1)) | (dt_diff.isna())
    g['run_seq'] = new_run.cumsum()
    return g

df_runs = df.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_run_ids)
df_runs = df_runs.reset_index(drop=True)

# Create run_id
df_runs['run_id'] = df_runs['cmdb_id'].astype(str) + '||' + df_runs['kpi_name'].astype(str) + '||' + df_runs['run_seq'].astype(str)

# Aggregate per run
run_agg = df_runs.groupby(['cmdb_id','kpi_name','run_id']).agg(
    run_start_timestamp = ('dt','min'),
    run_end_timestamp = ('dt','max'),
    points_in_run = ('dt','size'),
    run_max_value = ('value','max'),
    run_min_value = ('value','min'),
    global_p95 = ('p95','first'),
    global_p5  = ('p5','first')
).reset_index()

# duration_minutes inclusive
run_agg['duration_minutes'] = ((run_agg['run_end_timestamp'] - run_agg['run_start_timestamp']) / pd.Timedelta(minutes=1)).astype(int) + 1

# Compute per-run flags any_high / any_low from df_runs
df_runs['is_high'] = df_runs['value'] >= df_runs['p95']
df_runs['is_low']  = df_runs['value'] <= df_runs['p5']

run_flags = df_runs.groupby('run_id').agg(
    any_high = ('is_high','any'),
    any_low  = ('is_low','any')
).reset_index()

# Merge flags into run_agg
run_agg = run_agg.merge(run_flags, on='run_id', how='left')

# Assign breach_type robustly (object dtype)
def decide_breach_type(row):
    if row['any_high']:
        return 'high'
    if (not row['any_high']) and row['any_low']:
        return 'low'
    return np.nan

run_agg['breach_type'] = run_agg.apply(decide_breach_type, axis=1)

# Compute breach_ratio with rules
def compute_breach_ratio(row):
    p95 = row['global_p95']
    p5 = row['global_p5']
    maxv = row['run_max_value']
    minv = row['run_min_value']
    typ = row['breach_type']
    try:
        if typ == 'high':
            if pd.isna(p95):
                return np.nan
            if p95 > 0:
                return (maxv - p95) / p95
            else:
                if maxv > p95:
                    return large_inf
                else:
                    return 0.0
        elif typ == 'low':
            if pd.isna(p5):
                return np.nan
            if p5 > 0:
                return (p5 - minv) / p5
            else:
                if minv < p5:
                    return large_inf
                else:
                    return 0.0
        else:
            return np.nan
    except Exception:
        return np.nan

run_agg['breach_ratio'] = run_agg.apply(compute_breach_ratio, axis=1)
run_agg['breach_is_inf'] = run_agg['breach_ratio'] >= large_inf

# Filter runs: duration_minutes >= 2
run_agg_filtered = run_agg[run_agg['duration_minutes'] >= 2].copy()

# Further filter: breach_ratio >= 0.5 OR breach_is_inf
run_agg_filtered = run_agg_filtered[
    (run_agg_filtered['breach_ratio'] >= 0.5) | (run_agg_filtered['breach_is_inf'])
].copy()

# Prepare top 20 faulty runs sorted by breach_ratio desc then points_in_run desc
display_df = run_agg_filtered.copy()
display_df['breach_ratio_display'] = display_df['breach_ratio'].replace({large_inf: np.inf})

top20_faulty = display_df.sort_values(['breach_ratio_display','points_in_run'], ascending=[False, False]).head(20)

top20_display = top20_faulty[[
    'cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','duration_minutes',
    'points_in_run','breach_type','breach_ratio_display','global_p95','global_p5'
]].rename(columns={'breach_ratio_display':'breach_ratio'}).reset_index(drop=True)

# 2) Aggregate counts of retained runs by cmdb_id: top 10
agg_by_cm = run_agg_filtered.groupby('cmdb_id').agg(
    num_fault_runs = ('run_id','nunique'),
    total_fault_minutes = ('duration_minutes','sum'),
    earliest_run_start = ('run_start_timestamp','min'),
    latest_run_end = ('run_end_timestamp','max')
).reset_index()

top10_components_faults = agg_by_cm.sort_values('num_fault_runs', ascending=False).head(10).reset_index(drop=True)

# Compact outputs
top20_display, top10_components_faults

```
Out[4]:
```
```python
# Plain-English summary of faulty runs analysis (compact)
summary = (
    "Fault-run analysis (2022-03-20 15:00–15:30 UTC+8) — key findings:\n\n"
    "1) Many long consecutive anomalous runs were detected per (cmdb_id, kpi_name) after filtering\n"
    "   for runs >=2 minutes and breach_ratio >=0.5 (or +Inf). A large portion of top runs span the\n"
    "   entire 30-minute incident window (duration = 30 minutes, points_in_run = 30).\n\n"
    "2) Top faulty runs (examples) are dominated by 'high' breaches with breach_ratio = +Inf\n"
    "   (this occurs when global P95 or P5 is 0 and observed values exceed that level). Examples\n"
    "   include: adservice.runtime.jvm_threads_started and many container/node KPIs\n"
    "   (container_cpu_cfs_throttled_*, container_fs_reads/writes, container_memory_failures, etc.).\n\n"
    "3) Component-level aggregation: the largest counts of fault runs occurred on node-3 (11 runs,\n"
    "   120 total fault-minutes) and node-5 (6 runs, 138 total fault-minutes). Several service pods\n"
    "   also show many fault runs (checkoutservice-0/1/2, adservice/adservice2, frontend pods).\n\n"
    "4) Important caveat: many KPIs have zero or constant historical thresholds (P95==P5 or P95/P5==0),\n"
    "   which causes the whole per-minute series to be flagged anomalous. This likely reflects low-\n"
    "   variance or quantized metrics rather than transient faults. Treat +Inf breach_ratios cautiously.\n\n"
    "Recommended next steps:\n"
    "- Inspect raw KPI time series for top runs to confirm real deviation vs. quantized/constant metrics.\n"
    "- Cross-check node/pod logs and trace data for correlated errors during 15:00–15:30.\n"
    "- If thresholds are invalid for some KPIs, adjust thresholding method or exclude non-informative KPIs.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name       run_start_timestamp         run_end_timestamp  duration_minutes  points_in_run breach_type  breach_ratio  global_p95  global_p5
0           adservice                        runtime.jvm_threads_started 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
1        adservice2-0  container.node-6.container_cpu_cfs_throttled_p... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
2        adservice2-0  container.node-6.container_cpu_cfs_throttled_s... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
3       cartservice-2  container.node-5.container_cpu_cfs_throttled_p... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
4       cartservice-2  container.node-5.container_cpu_cfs_throttled_s... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
5   checkoutservice-0     container.node-5.container_fs_inodes./dev/vda1 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
6   checkoutservice-1       container.node-5.container_fs_reads./dev/vda 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
7   checkoutservice-1    container.node-5.container_fs_reads_MB./dev/vda 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
8   checkoutservice-1  container.node-5.container_memory_failures.con... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
9   checkoutservice-1  container.node-5.container_memory_failures.hie... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
10  checkoutservice-2       container.node-5.container_fs_reads./dev/vda 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
11  checkoutservice-2    container.node-5.container_fs_reads_MB./dev/vda 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
12  checkoutservice-2  container.node-5.container_memory_failures.con... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
13  checkoutservice-2  container.node-5.container_memory_failures.hie... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
14  currencyservice-1  container.node-5.container_cpu_cfs_throttled_p... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
15  currencyservice-1  container.node-5.container_cpu_cfs_throttled_s... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
16  currencyservice-2  container.node-5.container_cpu_cfs_throttled_p... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
17  currencyservice-2  container.node-5.container_cpu_cfs_throttled_s... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
18     emailservice-1  mesh.source.emailservice.jaeger-collector.isti... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0
19         frontend-0  container.node-5.container_cpu_cfs_throttled_s... 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00                30             30        high           inf         0.0        0.0,              cmdb_id  num_fault_runs  total_fault_minutes        earliest_run_start            latest_run_end
0             node-3              11                  120 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
1             node-2              10                   26 2022-03-20 15:01:00+08:00 2022-03-20 15:07:00+08:00
2  checkoutservice-0              10                  117 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
3             node-5               6                  138 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
4             node-1               5                   38 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
5  checkoutservice-1               5                  122 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
6  checkoutservice-2               4                  120 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
7     emailservice-1               4                   36 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
8  currencyservice-1               3                   62 2022-03-20 15:00:00+08:00 2022-03-20 15:29:00+08:00
9             node-6               3                   17 2022-03-20 15:00:00+08:00 2022-03-20 15:12:00+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure dt columns exist and are timezone-aware for trace_df, error_logs_df, log_df
for df in (trace_df, error_logs_df, log_df):
    if 'dt' not in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if df['dt'].dt.tz is None:
            df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive start, exclusive end)
start = pd.Timestamp('2022-03-20 15:00:00', tz=tz)
end   = pd.Timestamp('2022-03-20 15:30:00', tz=tz)

# 1) Aggregate trace metrics in window for error_rate and duration_p95
trace_win = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] < end)].copy()

# Select error_rate and duration_p95 traces
trace_error = trace_win[trace_win['trace_name'].str.contains('error_rate', na=False)].copy()
trace_dur95 = trace_win[trace_win['trace_name'].str.contains('duration_p95', na=False)].copy()

# Aggregate per minute
agg_error = trace_error.groupby('dt')['value'].sum().reset_index().rename(columns={'value':'total_error_rate'})
agg_dur95  = trace_dur95.groupby('dt')['value'].sum().reset_index().rename(columns={'value':'total_duration_p95'})

# Merge to one per-minute table (outer to keep minutes present in either)
trace_minute = pd.merge(agg_error, agg_dur95, on='dt', how='outer').fillna(0)
trace_minute = trace_minute.sort_values('total_error_rate', ascending=False).reset_index(drop=True)

# Top 10 timestamps by total_error_rate
top10_trace_minutes = trace_minute[['dt','total_error_rate','total_duration_p95']].rename(columns={'dt':'timestamp'}).head(10)

# 2) error_logs counts per minute in window, top 10 by message count
err_win = error_logs_df[(error_logs_df['dt'] >= start) & (error_logs_df['dt'] < end)].copy()
err_counts = err_win.groupby('dt').size().reset_index(name='message_count').sort_values('message_count', ascending=False).reset_index(drop=True)
top10_error_logs_minutes = err_counts.rename(columns={'dt':'timestamp'}).head(10)

# 3) Combined candidate peak minutes (union of minutes from both top-10 lists)
t1 = top10_trace_minutes[['timestamp','total_error_rate']].copy()
t2 = top10_error_logs_minutes[['timestamp','message_count']].copy()

combined = pd.merge(t1, t2, on='timestamp', how='outer').fillna(0)
combined = combined.sort_values(['total_error_rate','message_count'], ascending=[False, False]).reset_index(drop=True)
combined_top10 = combined.head(10)

# 4) Diagnostics for top 2 timestamps from combined table
top_candidate_ts = combined_top10['timestamp'].head(2).tolist()

# Recompute minute-level anomaly counts across all components from anomalies (ensure anomalies exists in kernel)
# anomalies: DataFrame of anomaly rows within window (from prior steps)
# Build minute-level anomaly counts per cmdb_id
minute_level_anomalies = anomalies.groupby(['dt','cmdb_id']).size().reset_index(name='anomaly_count')

# Prepare containers for diagnostics
diag_metric_anomalies = {}
diag_trace_error = {}
diag_trace_dur95 = {}
diag_error_logs_msgs = {}
diag_log_rows = {}

for ts in top_candidate_ts:
    # a) Top 10 cmdb_id by metric anomaly_count at that exact minute
    m = minute_level_anomalies[minute_level_anomalies['dt'] == ts].sort_values('anomaly_count', ascending=False).head(10)
    m = m.rename(columns={'dt':'timestamp'})[['timestamp','cmdb_id','anomaly_count']]
    diag_metric_anomalies[ts] = m

    # b) Top 10 trace rows at that timestamp for 'error_rate'
    te = trace_error[trace_error['dt'] == ts].sort_values('value', ascending=False).head(10)
    te = te[['trace_name','cmdb_id','value']].reset_index(drop=True)
    diag_trace_error[ts] = te

    # c) Top 10 trace rows at that timestamp for 'duration_p95'
    td = trace_dur95[trace_dur95['dt'] == ts].sort_values('value', ascending=False).head(10)
    td = td[['trace_name','cmdb_id','value']].reset_index(drop=True)
    diag_trace_dur95[ts] = td

    # d) Top 10 error_logs messages at that timestamp with counts
    em = err_win[err_win['dt'] == ts].groupby('message').size().reset_index(name='count').sort_values('count', ascending=False).head(10)
    em = em.rename(columns={'message':'message_sample'})
    diag_error_logs_msgs[ts] = em

    # e) Top 10 log.csv rows at that timestamp sorted by value desc
    logs_at_ts = log_df[(log_df['dt'] == ts)].sort_values('value', ascending=False).head(10)
    logs_at_ts = logs_at_ts[['cmdb_id','log_name','value']].reset_index(drop=True)
    diag_log_rows[ts] = logs_at_ts

# Keep outputs compact: show the aggregated tables and diagnostics dictionaries (each value is a small DF)
top10_trace_minutes, top10_error_logs_minutes, combined_top10, diag_metric_anomalies, diag_trace_error, diag_trace_dur95, diag_error_logs_msgs, diag_log_rows

```
Out[5]:
```
```python
# Plain-English compact summary of the findings (based on trace, error_logs, metrics diagnostics)
summary = (
    "Findings for 2022-03-20 15:00–15:30 (UTC+8):\n\n"
    "1) Trace aggregates:\n"
    "   - Summed 'error_rate' per minute is effectively zero for the top minutes in the window.\n"
    "   - 'duration_p95' shows notable peaks earlier in the window (examples: 15:06 ≈27.33s, 15:04 ≈21.60s, 15:05 ≈21.01s).\n\n"
    "2) Error-log peaks:\n"
    "   - Top error-log minutes are 15:24:01 and 15:26:57 (each with 4 messages). Several other minutes have 3 messages.\n\n"
    "3) Combined candidate minutes:\n"
    "   - The top candidate minutes (union of trace and log peaks) are 15:24:01 and 15:26:57 (both have message_count=4 and total_error_rate=0).\n"
    "   - These two minutes are the primary candidates for investigation based on error-log volume.\n\n"
    "4) Diagnostic snapshots at the two candidate minutes (15:24:01 and 15:26:57):\n"
    "   - Metric anomalies: no per-component anomaly counts at those exact minutes (empty results).\n"
    "   - Trace rows: no 'error_rate' or 'duration_p95' trace entries at those exact minute timestamps (empty results).\n"
    "   - Error logs: both minutes contain repeated warning message(s) (e.g. 'severity: warning, message: failed to retrieve...'), 4 occurrences each.\n"
    "   - log.csv: no log metric rows at those exact minutes.\n\n"
    "5) Interpretation & next steps:\n"
    "   - The strongest signals at 15:24:01 and 15:26:57 come from error logs, not from aggregated trace error_rate or per-minute metric anomalies.\n"
    "   - There are earlier trace duration spikes (~15:04–15:06) that may indicate latency issues; these do not align with the error-log minutes exactly.\n"
    "   - Recommended actions: inspect raw error-log messages and surrounding seconds (±1–2 minutes) around 15:24:01 and 15:26:57; correlate with traces/metrics in the expanded window (including the 15:04–15:06 duration_p95 peaks) to determine if these are related incidents or separate transient warnings.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp  total_error_rate  total_duration_p95
0 2022-03-20 15:00:00+08:00               0.0            1.720804
1 2022-03-20 15:01:00+08:00               0.0            1.962030
2 2022-03-20 15:02:00+08:00               0.0            7.714480
3 2022-03-20 15:03:00+08:00               0.0            8.398514
4 2022-03-20 15:04:00+08:00               0.0           21.599562
5 2022-03-20 15:05:00+08:00               0.0           21.011712
6 2022-03-20 15:06:00+08:00               0.0           27.331079
7 2022-03-20 15:07:00+08:00               0.0            1.430198
8 2022-03-20 15:08:00+08:00               0.0            1.708832
9 2022-03-20 15:09:00+08:00               0.0            1.498454,                   timestamp  message_count
0 2022-03-20 15:26:57+08:00              4
1 2022-03-20 15:24:01+08:00              4
2 2022-03-20 15:25:42+08:00              3
3 2022-03-20 15:25:40+08:00              3
4 2022-03-20 15:06:04+08:00              3
5 2022-03-20 15:25:58+08:00              3
6 2022-03-20 15:18:13+08:00              3
7 2022-03-20 15:14:18+08:00              3
8 2022-03-20 15:25:27+08:00              3
9 2022-03-20 15:11:12+08:00              3,                   timestamp  total_error_rate  message_count
0 2022-03-20 15:24:01+08:00               0.0            4.0
1 2022-03-20 15:26:57+08:00               0.0            4.0
2 2022-03-20 15:06:04+08:00               0.0            3.0
3 2022-03-20 15:11:12+08:00               0.0            3.0
4 2022-03-20 15:14:18+08:00               0.0            3.0
5 2022-03-20 15:18:13+08:00               0.0            3.0
6 2022-03-20 15:25:27+08:00               0.0            3.0
7 2022-03-20 15:25:40+08:00               0.0            3.0
8 2022-03-20 15:25:42+08:00               0.0            3.0
9 2022-03-20 15:25:58+08:00               0.0            3.0, {Timestamp('2022-03-20 15:24:01+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [timestamp, cmdb_id, anomaly_count]
Index: [], Timestamp('2022-03-20 15:26:57+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [timestamp, cmdb_id, anomaly_count]
Index: []}, {Timestamp('2022-03-20 15:24:01+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [trace_name, cmdb_id, value]
Index: [], Timestamp('2022-03-20 15:26:57+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [trace_name, cmdb_id, value]
Index: []}, {Timestamp('2022-03-20 15:24:01+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [trace_name, cmdb_id, value]
Index: [], Timestamp('2022-03-20 15:26:57+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [trace_name, cmdb_id, value]
Index: []}, {Timestamp('2022-03-20 15:24:01+0800', tz='Asia/Shanghai'):                                       message_sample  count
0  severity: warning, message: failed to retrieve...      4, Timestamp('2022-03-20 15:26:57+0800', tz='Asia/Shanghai'):                                       message_sample  count
0  severity: warning, message: failed to retrieve...      4}, {Timestamp('2022-03-20 15:24:01+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [cmdb_id, log_name, value]
Index: [], Timestamp('2022-03-20 15:26:57+0800', tz='Asia/Shanghai'): Empty DataFrame
Columns: [cmdb_id, log_name, value]
Index: []})```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse kernel variables: minute_level_anomalies, trace_df, error_logs_df, log_df
# Ensure dt columns exist and are timezone-aware
for df in (trace_df, error_logs_df, log_df, minute_level_anomalies):
    if 'dt' not in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if df['dt'].dt.tz is None:
            df['dt'] = pd.to_datetime(df['dt'], utc=True).dt.tz_convert(tz)

# Define windows (inclusive)
A_start = pd.Timestamp('2022-03-20 15:04:00', tz=tz)
A_end   = pd.Timestamp('2022-03-20 15:06:00', tz=tz)
B_start = pd.Timestamp('2022-03-20 15:23:00', tz=tz)
B_end   = pd.Timestamp('2022-03-20 15:27:00', tz=tz)

# Helper to produce all five outputs for a window
def window_diagnostics(window_name, start_ts, end_ts):
    # 1) Top 10 components by summed metric anomaly_count across the window
    m = minute_level_anomalies.copy()
    m_win = m[(m['dt'] >= start_ts) & (m['dt'] <= end_ts)].copy()
    comp_agg = m_win.groupby('cmdb_id').agg(
        summed_anomaly_count = ('anomaly_count','sum'),
        minutes_with_anomaly = ('dt', lambda x: x.nunique()),
        earliest_minute_with_anomaly = ('dt','min'),
        latest_minute_with_anomaly = ('dt','max')
    ).reset_index()
    comp_agg = comp_agg.sort_values('summed_anomaly_count', ascending=False).head(10)
    comp_agg.insert(0, 'window_name', window_name)
    comp_agg = comp_agg[[
        'window_name','cmdb_id','summed_anomaly_count','minutes_with_anomaly',
        'earliest_minute_with_anomaly','latest_minute_with_anomaly'
    ]].reset_index(drop=True)

    # 2) Top 10 trace rows with 'error_rate' in window
    t = trace_df.copy()
    te = t[(t['dt'] >= start_ts) & (t['dt'] <= end_ts) & (t['trace_name'].str.contains('error_rate', na=False))].copy()
    te = te.sort_values('value', ascending=False).head(10)
    te = te[['trace_name','cmdb_id','dt','value']].rename(columns={'dt':'timestamp'})
    te.insert(0,'window_name',window_name)
    te = te[['window_name','trace_name','cmdb_id','timestamp','value']].reset_index(drop=True)

    # 3) Top 10 trace rows with 'duration_p95' in window
    td = t[(t['dt'] >= start_ts) & (t['dt'] <= end_ts) & (t['trace_name'].str.contains('duration_p95', na=False))].copy()
    td = td.sort_values('value', ascending=False).head(10)
    td = td[['trace_name','cmdb_id','dt','value']].rename(columns={'dt':'timestamp'})
    td.insert(0,'window_name',window_name)
    td = td[['window_name','trace_name','cmdb_id','timestamp','value']].reset_index(drop=True)

    # 4) Top 10 error_logs messages within window aggregated by exact-second timestamp
    e = error_logs_df.copy()
    ew = e[(e['dt'] >= start_ts) & (e['dt'] <= end_ts)].copy()
    if 'message' in ew.columns:
        em = ew.groupby('message').agg(
            count = ('message','size'),
            earliest_timestamp = ('dt','min'),
            latest_timestamp = ('dt','max')
        ).reset_index().rename(columns={'message':'message_sample'})
        em = em.sort_values('count', ascending=False).head(10)
        em.insert(0,'window_name',window_name)
        em = em[['window_name','message_sample','count','earliest_timestamp','latest_timestamp']].reset_index(drop=True)
    else:
        em = pd.DataFrame(columns=['window_name','message_sample','count','earliest_timestamp','latest_timestamp'])

    # 5) Top 10 log.csv rows within window sorted by value desc (minute-granular)
    l = log_df.copy()
    lw = l[(l['dt'] >= start_ts) & (l['dt'] <= end_ts)].copy()
    if lw.shape[0] > 0:
        logs_top = lw.sort_values('value', ascending=False).head(10)
        logs_top = logs_top[['cmdb_id','log_name','dt','value']].rename(columns={'dt':'timestamp'})
        logs_top.insert(0,'window_name',window_name)
        logs_top = logs_top[['window_name','cmdb_id','log_name','timestamp','value']].reset_index(drop=True)
    else:
        logs_top = pd.DataFrame(columns=['window_name','cmdb_id','log_name','timestamp','value'])

    return comp_agg, te, td, em, logs_top

# Run diagnostics for Window A and B
A_comp, A_trace_err, A_trace_dur, A_err_logs, A_logs = window_diagnostics('A', A_start, A_end)
B_comp, B_trace_err, B_trace_dur, B_err_logs, B_logs = window_diagnostics('B', B_start, B_end)

# Compact outputs (only the described aggregated columns, max 10 rows each)
A_comp, A_trace_err, A_trace_dur, A_err_logs, A_logs, B_comp, B_trace_err, B_trace_dur, B_err_logs, B_logs

```
Out[6]:
```
```python
# Compact plain-English summary for Windows A and B (based on kernel diagnostics)
summary = (
    "Diagnostic summary — Window A (2022-03-20 15:04–15:06 UTC+8) vs Window B (2022-03-20 15:23–15:27 UTC+8):\n\n"
    "Window A (trace-latency cluster, 15:04–15:06):\n"
    "- Metrics: High summed anomaly counts concentrated across many services. Top components by summed anomalies: "
    "adservice (915 over 3 minutes), adservice2 (893), frontend2-0 (324), frontend-1 (283), frontend-0 (280). "
    "All top components show anomalies in all 3 minutes (15:04–15:06).\n"
    "- Traces: 'duration_p95' shows clear latency spikes (~5.04s at 15:04; ~4.66s at 15:05; ~4.84s at 15:06) "
    "in traces involving frontends -> checkoutservice (examples: trace.to_checkoutservice-0.duration_p95, trace.from_frontend-0.duration_p95).\n"
    "- trace 'error_rate' entries in the window are present but their aggregated values are 0.0 (no trace-level error_rate spike).\n"
    "- Logs & error messages: The dominant error-log message is 'severity: warning, message: failed to retrieve...' (126 occurrences in the window). "
    "log.row_count is high for frontend pods (many minutes >1,400–2,000), indicating heavy logging activity during the latency spikes.\n"
    "- Interpretation: Window A is characterized by pronounced latency (p95) spikes correlated with broad metric anomalies and many warning-log entries. "
    "This looks like a latency incident affecting frontend→checkoutservice paths and many services concurrently.\n\n"
    "Window B (error-log cluster, 15:23–15:27):\n"
    "- Metrics: Even larger anomaly totals across the same major components. Top components: adservice (1533 over 5 minutes), adservice2 (1472), "
    "frontend2-0 (550), frontend-0 (530), frontend-1 (474). Anomalies persist across all 5 minutes (15:23–15:27).\n"
    "- Traces: 'error_rate' traces again show values of 0.0 in the minutes shown. 'duration_p95' traces exist but with small values (~0.10–0.12), "
    "much lower than Window A's multi-second spikes.\n"
    "- Logs & error messages: The same warning message ('failed to retrieve...') dominates with ~267 occurrences in this window. frontend log.row_count "
    "is high (top entries ~1,700–1,997), indicating heavy activity and many warnings.\n"
    "- Interpretation: Window B is dominated by error-log volume and heavy metric anomalies across many services but without large trace duration spikes. "
    "This suggests broad functional/warning errors or noisy logs rather than a pure latency spike.\n\n"
    "Overall conclusions and actionables:\n"
    "- There are two distinct clusters: (A) a short latency spike cluster (15:04–15:06) with multi-second p95s and correlated metric anomalies, "
    "and (B) a later, larger error-log cluster (15:23–15:27) with many warning messages and extensive metric anomalies but no large trace p95 spikes.\n"
    "- trace.error_rate is not a clear signal (values are 0.0); primary signals came from duration_p95 (latency) and from logs/metric anomalies.\n"
    "- Next steps: inspect raw traces and span details around 15:04–15:06 for frontend→checkoutservice paths, and inspect the full error-log contents and frontend logs "
    "for 15:23–15:27 (and surrounding minutes) to determine root cause. Also review node/pod-level metrics (CPU/memory/disk) for nodes showing many anomalous KPIs."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  window_name                  cmdb_id  summed_anomaly_count  minutes_with_anomaly earliest_minute_with_anomaly latest_minute_with_anomaly
0           A                adservice                   915                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
1           A               adservice2                   893                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
2           A              frontend2-0                   324                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
3           A               frontend-1                   283                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
4           A               frontend-0                   280                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
5           A               frontend-2                   273                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
6           A  productcatalogservice-0                   257                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
7           A  productcatalogservice-2                   246                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
8           A        checkoutservice-1                   239                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00
9           A        checkoutservice-0                   223                     3    2022-03-20 15:04:00+08:00  2022-03-20 15:06:00+08:00,   window_name                         trace_name             cmdb_id                 timestamp  value
0           A  trace.from_frontend2-0.error_rate  shippingservice2-0 2022-03-20 15:06:00+08:00    0.0
1           A   trace.from_frontend-0.error_rate         adservice-0 2022-03-20 15:04:00+08:00    0.0
2           A   trace.from_frontend-1.error_rate         adservice-0 2022-03-20 15:04:00+08:00    0.0
3           A   trace.from_frontend-2.error_rate         adservice-0 2022-03-20 15:04:00+08:00    0.0
4           A   trace.from_frontend-0.error_rate         adservice-1 2022-03-20 15:04:00+08:00    0.0
5           A   trace.from_frontend-1.error_rate         adservice-1 2022-03-20 15:04:00+08:00    0.0
6           A   trace.from_frontend-2.error_rate         adservice-1 2022-03-20 15:04:00+08:00    0.0
7           A   trace.from_frontend-0.error_rate         adservice-2 2022-03-20 15:04:00+08:00    0.0
8           A   trace.from_frontend-1.error_rate         adservice-2 2022-03-20 15:04:00+08:00    0.0
9           A   trace.from_frontend-2.error_rate         adservice-2 2022-03-20 15:04:00+08:00    0.0,   window_name                               trace_name            cmdb_id                 timestamp     value
0           A  trace.to_checkoutservice-0.duration_p95         frontend-0 2022-03-20 15:04:00+08:00  5.038030
1           A       trace.from_frontend-0.duration_p95  checkoutservice-0 2022-03-20 15:04:00+08:00  5.038030
2           A  trace.to_checkoutservice-0.duration_p95         frontend-2 2022-03-20 15:06:00+08:00  4.844970
3           A       trace.from_frontend-2.duration_p95  checkoutservice-0 2022-03-20 15:06:00+08:00  4.844970
4           A  trace.to_checkoutservice-0.duration_p95         frontend-2 2022-03-20 15:05:00+08:00  4.665053
5           A       trace.from_frontend-2.duration_p95  checkoutservice-0 2022-03-20 15:05:00+08:00  4.665053
6           A       trace.from_frontend-1.duration_p95  checkoutservice-0 2022-03-20 15:05:00+08:00  4.499961
7           A  trace.to_checkoutservice-0.duration_p95         frontend-1 2022-03-20 15:05:00+08:00  4.499961
8           A  trace.to_checkoutservice-0.duration_p95         frontend-0 2022-03-20 15:06:00+08:00  3.780066
9           A       trace.from_frontend-0.duration_p95  checkoutservice-0 2022-03-20 15:06:00+08:00  3.780066,   window_name                                     message_sample  count        earliest_timestamp          latest_timestamp
0           A  severity: warning, message: failed to retrieve...    126 2022-03-20 15:04:01+08:00 2022-03-20 15:05:59+08:00,   window_name      cmdb_id       log_name                 timestamp   value
0           A   frontend-2  log.row_count 2022-03-20 15:06:00+08:00  2014.0
1           A   frontend-0  log.row_count 2022-03-20 15:06:00+08:00  1883.0
2           A   frontend-2  log.row_count 2022-03-20 15:04:00+08:00  1826.0
3           A   frontend-0  log.row_count 2022-03-20 15:04:00+08:00  1714.0
4           A   frontend-0  log.row_count 2022-03-20 15:05:00+08:00  1713.0
5           A   frontend-2  log.row_count 2022-03-20 15:05:00+08:00  1645.0
6           A  adservice-0  log.row_count 2022-03-20 15:05:00+08:00  1590.0
7           A   frontend-1  log.row_count 2022-03-20 15:05:00+08:00  1476.0
8           A   frontend-1  log.row_count 2022-03-20 15:06:00+08:00  1427.0
9           A  adservice-0  log.row_count 2022-03-20 15:06:00+08:00  1418.0,   window_name                  cmdb_id  summed_anomaly_count  minutes_with_anomaly earliest_minute_with_anomaly latest_minute_with_anomaly
0           B                adservice                  1533                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
1           B               adservice2                  1472                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
2           B              frontend2-0                   550                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
3           B               frontend-0                   530                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
4           B               frontend-1                   474                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
5           B  productcatalogservice-2                   400                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
6           B               frontend-2                   395                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
7           B        checkoutservice-2                   394                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
8           B  productcatalogservice-1                   386                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00
9           B  productcatalogservice-0                   378                     5    2022-03-20 15:23:00+08:00  2022-03-20 15:27:00+08:00,   window_name                         trace_name             cmdb_id                 timestamp  value
0           B  trace.from_frontend2-0.error_rate  shippingservice2-0 2022-03-20 15:27:00+08:00    0.0
1           B   trace.from_frontend-0.error_rate         adservice-0 2022-03-20 15:23:00+08:00    0.0
2           B   trace.from_frontend-1.error_rate         adservice-0 2022-03-20 15:23:00+08:00    0.0
3           B   trace.from_frontend-2.error_rate         adservice-0 2022-03-20 15:23:00+08:00    0.0
4           B   trace.from_frontend-0.error_rate         adservice-1 2022-03-20 15:23:00+08:00    0.0
5           B   trace.from_frontend-1.error_rate         adservice-1 2022-03-20 15:23:00+08:00    0.0
6           B   trace.from_frontend-2.error_rate         adservice-1 2022-03-20 15:23:00+08:00    0.0
7           B   trace.from_frontend-0.error_rate         adservice-2 2022-03-20 15:23:00+08:00    0.0
8           B   trace.from_frontend-1.error_rate         adservice-2 2022-03-20 15:23:00+08:00    0.0
9           B   trace.from_frontend-2.error_rate         adservice-2 2022-03-20 15:23:00+08:00    0.0,   window_name                                trace_name             cmdb_id                 timestamp     value
0           B              trace.from_root.duration_p95         frontend2-0 2022-03-20 15:26:00+08:00  0.121300
1           B              trace.from_root.duration_p95         frontend2-0 2022-03-20 15:24:00+08:00  0.112013
2           B              trace.from_root.duration_p95         frontend2-0 2022-03-20 15:23:00+08:00  0.110231
3           B              trace.from_root.duration_p95         frontend2-0 2022-03-20 15:27:00+08:00  0.107847
4           B              trace.from_root.duration_p95         frontend2-0 2022-03-20 15:25:00+08:00  0.105704
5           B       trace.from_frontend2-0.duration_p95  checkoutservice2-0 2022-03-20 15:23:00+08:00  0.101431
6           B  trace.to_checkoutservice2-0.duration_p95         frontend2-0 2022-03-20 15:23:00+08:00  0.101431
7           B       trace.from_frontend2-0.duration_p95  checkoutservice2-0 2022-03-20 15:27:00+08:00  0.099924
8           B  trace.to_checkoutservice2-0.duration_p95         frontend2-0 2022-03-20 15:27:00+08:00  0.099924
9           B        trace.from_frontend-2.duration_p95   checkoutservice-1 2022-03-20 15:27:00+08:00  0.095512,   window_name                                     message_sample  count        earliest_timestamp          latest_timestamp
0           B  severity: warning, message: failed to retrieve...    267 2022-03-20 15:23:00+08:00 2022-03-20 15:27:00+08:00,   window_name     cmdb_id       log_name                 timestamp   value
0           B  frontend-2  log.row_count 2022-03-20 15:25:00+08:00  1997.0
1           B  frontend-0  log.row_count 2022-03-20 15:26:00+08:00  1929.0
2           B  frontend-0  log.row_count 2022-03-20 15:23:00+08:00  1902.0
3           B  frontend-0  log.row_count 2022-03-20 15:25:00+08:00  1889.0
4           B  frontend-2  log.row_count 2022-03-20 15:24:00+08:00  1820.0
5           B  frontend-2  log.row_count 2022-03-20 15:27:00+08:00  1809.0
6           B  frontend-0  log.row_count 2022-03-20 15:24:00+08:00  1805.0
7           B  frontend-0  log.row_count 2022-03-20 15:27:00+08:00  1798.0
8           B  frontend-2  log.row_count 2022-03-20 15:23:00+08:00  1764.0
9           B  frontend-2  log.row_count 2022-03-20 15:26:00+08:00  1677.0)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
large_inf = 1e9

# Ensure dt columns are present and tz-aware
for df in (metric_df, window_merged, trace_df, log_df, error_logs_df):
    if 'dt' not in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if df['dt'].dt.tz is None:
            df['dt'] = pd.to_datetime(df['dt'], utc=True).dt.tz_convert(tz)

candidates = {
    'checkoutservice-0_A': {
        'cmdb_ids': ['checkoutservice-0'],
        'start': pd.Timestamp('2022-03-20 15:04:00', tz=tz),
        'end':   pd.Timestamp('2022-03-20 15:06:00', tz=tz)
    },
    'adservice_B': {
        'cmdb_ids': ['adservice','adservice-0','adservice-1','adservice-2'],
        'start': pd.Timestamp('2022-03-20 15:23:00', tz=tz),
        'end':   pd.Timestamp('2022-03-20 15:27:00', tz=tz)
    },
    'adservice-0_B': {
        'cmdb_ids': ['adservice-0'],
        'start': pd.Timestamp('2022-03-20 15:23:00', tz=tz),
        'end':   pd.Timestamp('2022-03-20 15:27:00', tz=tz)
    },
    'adservice-1_B': {
        'cmdb_ids': ['adservice-1'],
        'start': pd.Timestamp('2022-03-20 15:23:00', tz=tz),
        'end':   pd.Timestamp('2022-03-20 15:27:00', tz=tz)
    },
    'adservice-2_B': {
        'cmdb_ids': ['adservice-2'],
        'start': pd.Timestamp('2022-03-20 15:23:00', tz=tz),
        'end':   pd.Timestamp('2022-03-20 15:27:00', tz=tz)
    }
}

def compute_breach_ratio_from_stats(p95, p5, maxv, minv):
    # compute high_ratio and low_ratio as described, return max
    high_ratio = 0.0
    low_ratio = 0.0
    # high
    try:
        if pd.notna(p95) and maxv >= p95:
            if p95 > 0:
                high_ratio = (maxv - p95) / p95
            else:
                high_ratio = large_inf if maxv > p95 else 0.0
    except:
        high_ratio = 0.0
    # low
    try:
        if pd.notna(p5) and minv <= p5:
            if p5 > 0:
                low_ratio = (p5 - minv) / p5
            else:
                low_ratio = large_inf if minv < p5 else 0.0
    except:
        low_ratio = 0.0
    return max(high_ratio, low_ratio)

outputs = {}

for name, meta in candidates.items():
    cmdb_ids = meta['cmdb_ids']
    start = meta['start']
    end = meta['end']

    # Filter windowed metric rows for these cmdb_ids
    w = window_merged[
        (window_merged['dt'] >= start) & (window_merged['dt'] <= end) &
        (window_merged['cmdb_id'].isin(cmdb_ids))
    ].copy()

    if w.shape[0] == 0:
        # produce empty tables with correct columns
        metric_top10_by_count = pd.DataFrame(columns=[
            'kpi_name','global_p95','global_p5','total_points_in_window','anomaly_count_in_window',
            'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window',
            'run_max_value_in_window','run_min_value_in_window'
        ])
        metric_top10_by_severity = metric_top10_by_count.copy()
    else:
        # general stats per kpi_name
        gen = w.groupby('kpi_name').agg(
            global_p95 = ('p95','max'),
            global_p5  = ('p5','min'),
            total_points_in_window = ('value','size'),
            run_max_value_in_window = ('value','max'),
            run_min_value_in_window = ('value','min')
        ).reset_index()

        # anomaly-specific stats
        anomaly_rows = w[w['anomaly'] == True].copy()
        if anomaly_rows.shape[0] > 0:
            an = anomaly_rows.groupby('kpi_name').agg(
                anomaly_count_in_window = ('anomaly','size'),
                earliest_anomaly_timestamp_in_window = ('dt','min'),
                latest_anomaly_timestamp_in_window = ('dt','max')
            ).reset_index()
        else:
            an = pd.DataFrame(columns=['kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])

        kpi_agg = gen.merge(an, on='kpi_name', how='left')
        kpi_agg['anomaly_count_in_window'] = kpi_agg['anomaly_count_in_window'].fillna(0).astype(int)
        # compute breach_ratio_max per kpi using run stats and global thresholds
        kpi_agg['breach_ratio_max'] = kpi_agg.apply(
            lambda r: compute_breach_ratio_from_stats(r['global_p95'], r['global_p5'], r['run_max_value_in_window'], r['run_min_value_in_window']),
            axis=1
        )
        # 1) top 10 by anomaly count
        metric_top10_by_count = kpi_agg.sort_values('anomaly_count_in_window', ascending=False).head(10).reset_index(drop=True)[[
            'kpi_name','global_p95','global_p5','total_points_in_window','anomaly_count_in_window',
            'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window',
            'run_max_value_in_window','run_min_value_in_window'
        ]]

        # 2) severity score and top 10 by severity
        kpi_agg['severity_score'] = kpi_agg['anomaly_count_in_window'] * kpi_agg['breach_ratio_max'].clip(lower=1.0)
        # display breach_ratio as inf if large_inf
        kpi_agg['breach_ratio_display'] = kpi_agg['breach_ratio_max'].replace({large_inf: np.inf})
        metric_top10_by_severity = kpi_agg.sort_values('severity_score', ascending=False).head(10).reset_index(drop=True)[[
            'kpi_name','global_p95','global_p5','total_points_in_window','anomaly_count_in_window',
            'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window',
            'run_max_value_in_window','run_min_value_in_window','severity_score','breach_ratio_display'
        ]]

    # 3) Traces for the cmdb_ids in window with duration_p95 or error_rate
    t = trace_df[
        (trace_df['dt'] >= start) & (trace_df['dt'] <= end) &
        (trace_df['cmdb_id'].isin(cmdb_ids)) &
        (trace_df['trace_name'].str.contains('duration_p95|error_rate', na=False))
    ].copy()
    t_top10 = t.sort_values('value', ascending=False).head(10)[['trace_name','cmdb_id','dt','value']].rename(columns={'dt':'timestamp'}).reset_index(drop=True)

    # 4a) log.csv entries within window for these cmdb_ids sorted by value desc
    lwin = log_df[
        (log_df['dt'] >= start) & (log_df['dt'] <= end) &
        (log_df['cmdb_id'].isin(cmdb_ids))
    ].copy()
    logs_top10 = lwin.sort_values('value', ascending=False).head(10)[['cmdb_id','log_name','dt','value']].rename(columns={'dt':'timestamp'}).reset_index(drop=True)

    # 4b) error_logs aggregation by message within window for these cmdb_ids
    ewin = error_logs_df[
        (error_logs_df['dt'] >= start) & (error_logs_df['dt'] <= end) &
        (error_logs_df['cmdb_id'].isin(cmdb_ids))
    ].copy()
    if 'message' in ewin.columns and ewin.shape[0] > 0:
        e_agg = ewin.groupby('message').agg(
            count = ('message','size'),
            earliest_timestamp = ('dt','min'),
            latest_timestamp = ('dt','max')
        ).reset_index().rename(columns={'message':'message_sample'}).sort_values('count', ascending=False).head(10).reset_index(drop=True)
    else:
        e_agg = pd.DataFrame(columns=['message_sample','count','earliest_timestamp','latest_timestamp'])

    outputs[name] = {
        'metric_top10_by_count': metric_top10_by_count,
        'metric_top10_by_severity': metric_top10_by_severity if 'metric_top10_by_severity' in locals() else pd.DataFrame(),
        'trace_top10': t_top10,
        'logs_top10': logs_top10,
        'error_logs_top10': e_agg
    }

# Expose compact variables for the kernel (each <=10 rows)
checkout_metric_top10 = outputs['checkoutservice-0_A']['metric_top10_by_count']
checkout_metric_severity = outputs['checkoutservice-0_A']['metric_top10_by_severity']
checkout_trace_top10 = outputs['checkoutservice-0_A']['trace_top10']
checkout_logs_top10 = outputs['checkoutservice-0_A']['logs_top10']
checkout_error_logs_top10 = outputs['checkoutservice-0_A']['error_logs_top10']

adservice_metric_top10 = outputs['adservice_B']['metric_top10_by_count']
adservice_metric_severity = outputs['adservice_B']['metric_top10_by_severity']
adservice_trace_top10 = outputs['adservice_B']['trace_top10']
adservice_logs_top10 = outputs['adservice_B']['logs_top10']
adservice_error_logs_top10 = outputs['adservice_B']['error_logs_top10']

ad0_metric_top10 = outputs['adservice-0_B']['metric_top10_by_count']
ad0_metric_severity = outputs['adservice-0_B']['metric_top10_by_severity']
ad0_trace_top10 = outputs['adservice-0_B']['trace_top10']
ad0_logs_top10 = outputs['adservice-0_B']['logs_top10']
ad0_error_logs_top10 = outputs['adservice-0_B']['error_logs_top10']

ad1_metric_top10 = outputs['adservice-1_B']['metric_top10_by_count']
ad1_metric_severity = outputs['adservice-1_B']['metric_top10_by_severity']
ad1_trace_top10 = outputs['adservice-1_B']['trace_top10']
ad1_logs_top10 = outputs['adservice-1_B']['logs_top10']
ad1_error_logs_top10 = outputs['adservice-1_B']['error_logs_top10']

ad2_metric_top10 = outputs['adservice-2_B']['metric_top10_by_count']
ad2_metric_severity = outputs['adservice-2_B']['metric_top10_by_severity']
ad2_trace_top10 = outputs['adservice-2_B']['trace_top10']
ad2_logs_top10 = outputs['adservice-2_B']['logs_top10']
ad2_error_logs_top10 = outputs['adservice-2_B']['error_logs_top10']

# Return compact aggregates (kernel will display these variables)
checkout_metric_top10, checkout_metric_severity, checkout_trace_top10, checkout_logs_top10, checkout_error_logs_top10, \
adservice_metric_top10, adservice_metric_severity, adservice_trace_top10, adservice_logs_top10, adservice_error_logs_top10, \
ad0_metric_top10, ad0_metric_severity, ad0_trace_top10, ad0_logs_top10, ad0_error_logs_top10, \
ad1_metric_top10, ad1_metric_severity, ad1_trace_top10, ad1_logs_top10, ad1_error_logs_top10, \
ad2_metric_top10, ad2_metric_severity, ad2_trace_top10, ad2_logs_top10, ad2_error_logs_top10

```
Out[7]:
```
```python
# Plain-English compact summary for focused root-cause check (candidates + windows)
summary = (
    "Focused root-cause summary (compact):\n\n"
    "Candidate A — checkoutservice-0 (Window A: 2022-03-20 15:04–15:06 UTC+8):\n"
    "- Earliest anomalous minute: 15:04. Strong trace evidence: multiple duration_p95 spikes (≈5.04s at 15:04, ~4.66s at 15:05, ~4.84s at 15:06) "
    "for frontend→checkoutservice spans.\n"
    "- Metric evidence: many container/node KPIs (node-5 related: fs, IO, cpu_load_average, memory_mapped_file) flagged as anomalies across the 3-minute window. "
    "One high-severity metric is container_memory_mapped_file (large run_max) and many fs/io KPIs were consistently anomalous (though many show zero P95/P5, treat those cautiously).\n"
    "- Logs: checkoutservice log.row_count rose in the window; no error_log samples found for checkoutservice-0.\n"
    "- Likely cause: a latency/IO-related issue impacting checkoutservice-0 (frontends experienced high p95 to checkoutservice). Possible causes to inspect: node-5 disk I/O or container memory-mapped file growth affecting service latency. Verify raw fs/io and memory traces and pod resource limits on node-5.\n\n"
    "Candidate B — adservice (service) and pods adservice-0/1/2 (Window B: 2022-03-20 15:23–15:27 UTC+08):\n"
    "- Earliest anomalous minute: 15:23. Metric anomalies are widespread for adservice (service-level and all pods) across the 5-minute window; summed anomaly counts are highest for adservice/adservice pods.\n"
    "- Trace evidence: no large duration_p95 spikes (trace durations are very small ~0.00002–0.00003 in this window) and trace.error_rate values are ~0.0 — so traces do not show large latency or trace-level error-rate spikes.\n"
    "- Logs & error messages: heavy logging on adservice-0 (high log.row_count) and the global warning message ('severity: warning, message: failed to retrieve...') is very frequent in the B window (many occurrences across components). However per-pod error_logs for adservice pods were empty in our per-component extract — main sign is elevated log volume and warning messages elsewhere.\n"
    "- KPI patterns: several mesh/istio metrics related to requests/responses to adservice are anomalous (many zeros). Node/container KPIs flagged include CPU throttling/file-descriptor/inode/fs metrics (mostly with zero or constant thresholds). This pattern suggests functional request failures or mesh-level request disruptions rather than raw CPU/memory spikes.\n"
    "- Likely cause: a functional or network/mesh-related failure (request failures or service-call failures) affecting adservice and causing heavy warnings and increased log volume. Investigate service mesh (Istio) metrics and error messages, upstream dependencies, and network connectivity for nodes hosting adservice pods (also check node-5 related KPIs shown anomalous).\n\n"
    "Overall recommendation (next steps):\n"
    "1) For checkoutservice-0 (Window A): examine node-5 disk IO and container memory mappings, inspect pod CPU/memory throttling and raw fs/io time series, and review detailed spans for frontend→checkoutservice requests around 15:04–15:06.\n"
    "2) For adservice/adservice-* (Window B): inspect adservice pod logs (especially adservice-0), check Istio/mesh request/response metrics and retries, and investigate any upstream dependency or network issues starting at ~15:23.\n"
    "3) Caution: many KPI thresholds are zero/constant (P95==P5 or zero), so treat those automatic anomalies carefully — prioritize high-severity metrics with meaningful breach_ratio or direct trace/log corroboration.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name     global_p95      global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window
0    container.node-5.container_cpu_load_average_10s       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000
1  container.node-5.container_fs_io_time_weighted...       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000
2  container.node-5.container_fs_io_current./dev/...       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000
3  container.node-5.container_fs_io_time_seconds....       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000
4     container.node-5.container_fs_inodes./dev/vda1       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000
5  container.node-5.container_fs_inodes_free./dev...       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000
6   container.node-5.container_fs_limit_MB./dev/vda1  604630.738281  604630.738281                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00            604630.738281            604630.738281
7      container.node-5.container_fs_writes./dev/vda       1.500000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 1.500000                 0.000000
8     container.node-5.container_fs_writes./dev/vda1       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000
9  container.node-5.container_fs_sector_writes./d...       0.000000       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00                 0.000000                 0.000000,                                             kpi_name    global_p95      global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window  severity_score  breach_ratio_display
0      container.node-5.container_memory_mapped_file  2.636288e+06       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             1.020723e+07                 0.000000        8.615459               2.87182
1    container.node-5.container_cpu_load_average_10s  0.000000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             0.000000e+00                 0.000000        3.000000               0.00000
2  container.node-5.container_fs_io_current./dev/...  0.000000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             0.000000e+00                 0.000000        3.000000               0.00000
3  container.node-5.container_fs_io_time_seconds....  0.000000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             0.000000e+00                 0.000000        3.000000               0.00000
4     container.node-5.container_fs_inodes./dev/vda1  0.000000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             0.000000e+00                 0.000000        3.000000               0.00000
5  container.node-5.container_fs_io_time_weighted...  0.000000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             0.000000e+00                 0.000000        3.000000               0.00000
6   container.node-5.container_fs_limit_MB./dev/vda1  6.046307e+05  604630.738281                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             6.046307e+05            604630.738281        3.000000               0.00000
7      container.node-5.container_fs_writes./dev/vda  1.500000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             1.500000e+00                 0.000000        3.000000               0.00000
8     container.node-5.container_fs_writes./dev/vda1  0.000000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             0.000000e+00                 0.000000        3.000000               0.00000
9  container.node-5.container_fs_sector_writes./d...  0.000000e+00       0.000000                       3                        3            2022-03-20 15:04:00+08:00          2022-03-20 15:06:00+08:00             0.000000e+00                 0.000000        3.000000               0.00000,                            trace_name            cmdb_id                 timestamp     value
0  trace.from_frontend-0.duration_p95  checkoutservice-0 2022-03-20 15:04:00+08:00  5.038030
1  trace.from_frontend-2.duration_p95  checkoutservice-0 2022-03-20 15:06:00+08:00  4.844970
2  trace.from_frontend-2.duration_p95  checkoutservice-0 2022-03-20 15:05:00+08:00  4.665053
3  trace.from_frontend-1.duration_p95  checkoutservice-0 2022-03-20 15:05:00+08:00  4.499961
4  trace.from_frontend-0.duration_p95  checkoutservice-0 2022-03-20 15:06:00+08:00  3.780066
5  trace.from_frontend-1.duration_p95  checkoutservice-0 2022-03-20 15:06:00+08:00  3.599996
6  trace.from_frontend-2.duration_p95  checkoutservice-0 2022-03-20 15:04:00+08:00  2.613555
7  trace.from_frontend-1.duration_p95  checkoutservice-0 2022-03-20 15:04:00+08:00  1.703735
8        trace.from_root.duration_p95  checkoutservice-0 2022-03-20 15:04:00+08:00  0.774821
9             trace.self.duration_p95  checkoutservice-0 2022-03-20 15:04:00+08:00  0.740044,              cmdb_id         log_name                 timestamp  value
0  checkoutservice-0    log.row_count 2022-03-20 15:06:00+08:00  105.0
1  checkoutservice-0    log.row_count 2022-03-20 15:05:00+08:00   72.0
2  checkoutservice-0    log.row_count 2022-03-20 15:04:00+08:00   63.0
3  checkoutservice-0  log.error_count 2022-03-20 15:04:00+08:00    0.0
4  checkoutservice-0  log.error_count 2022-03-20 15:05:00+08:00    0.0
5  checkoutservice-0  log.error_count 2022-03-20 15:06:00+08:00    0.0, Empty DataFrame
Columns: [message_sample, count, earliest_timestamp, latest_timestamp]
Index: [],                                             kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window
0  mesh.destination.frontend.adservice.istio_requ...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
1  mesh.destination.frontend.adservice.istio_resp...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
2  mesh.destination.frontend.adservice.istio_requ...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
3         container.node-5.container_spec_cpu_period    100000.0   100000.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 100000.0                 100000.0
4  container.node-5.container_network_transmit_pa...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
5  container.node-5.container_network_transmit_er...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
6  container.node-5.container_tasks_state.uninter...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
7     container.node-5.container_tasks_state.stopped         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
8             container.node-5.container_threads_max         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
9     container.node-5.container_tasks_state.running         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0,                                             kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window  severity_score  breach_ratio_display
0  mesh.destination.frontend.adservice.istio_requ...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
1  mesh.destination.frontend.adservice.istio_resp...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
2  mesh.destination.frontend.adservice.istio_requ...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
3         container.node-5.container_spec_cpu_period    100000.0   100000.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 100000.0                 100000.0            15.0                   0.0
4  container.node-5.container_network_transmit_pa...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
5  container.node-5.container_network_transmit_er...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
6  container.node-5.container_tasks_state.uninter...         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
7     container.node-5.container_tasks_state.stopped         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
8             container.node-5.container_threads_max         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0
9     container.node-5.container_tasks_state.running         0.0        0.0                      15                       15            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0            15.0                   0.0,                            trace_name      cmdb_id                 timestamp     value
0  trace.from_frontend-0.duration_p95  adservice-0 2022-03-20 15:26:00+08:00  0.000028
1  trace.from_frontend-2.duration_p95  adservice-2 2022-03-20 15:25:00+08:00  0.000026
2  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:24:00+08:00  0.000023
3  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:26:00+08:00  0.000022
4  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:25:00+08:00  0.000021
5  trace.from_frontend-2.duration_p95  adservice-1 2022-03-20 15:27:00+08:00  0.000021
6  trace.from_frontend-2.duration_p95  adservice-1 2022-03-20 15:23:00+08:00  0.000021
7  trace.from_frontend-2.duration_p95  adservice-0 2022-03-20 15:25:00+08:00  0.000020
8  trace.from_frontend-2.duration_p95  adservice-0 2022-03-20 15:27:00+08:00  0.000020
9  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:23:00+08:00  0.000020,        cmdb_id       log_name                 timestamp   value
0  adservice-0  log.row_count 2022-03-20 15:24:00+08:00  1560.0
1  adservice-0  log.row_count 2022-03-20 15:26:00+08:00  1526.0
2  adservice-0  log.row_count 2022-03-20 15:27:00+08:00  1468.0
3  adservice-0  log.row_count 2022-03-20 15:25:00+08:00  1456.0
4  adservice-0  log.row_count 2022-03-20 15:23:00+08:00  1348.0
5  adservice-2  log.row_count 2022-03-20 15:24:00+08:00   148.0
6  adservice-1  log.row_count 2022-03-20 15:24:00+08:00   144.0
7  adservice-1  log.row_count 2022-03-20 15:27:00+08:00   136.0
8  adservice-1  log.row_count 2022-03-20 15:26:00+08:00   136.0
9  adservice-2  log.row_count 2022-03-20 15:26:00+08:00   134.0, Empty DataFrame
Columns: [message_sample, count, earliest_timestamp, latest_timestamp]
Index: [],                                             kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window
0  container.node-5.container_cpu_cfs_throttled_p...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
1  container.node-5.container_cpu_cfs_throttled_s...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
2    container.node-5.container_cpu_load_average_10s         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
3  container.node-5.container_fs_inodes_free./dev...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
4     container.node-5.container_fs_inodes./dev/vda1         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
5        container.node-5.container_file_descriptors        96.0       95.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                     95.0                     95.0
6  container.node-5.container_fs_reads_merged./de...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
7       container.node-5.container_fs_reads./dev/vda         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
8      container.node-5.container_fs_reads./dev/vda1         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
9  container.node-5.container_fs_io_current./dev/...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0,                                             kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window  severity_score  breach_ratio_display
0  container.node-5.container_cpu_cfs_throttled_p...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
1  container.node-5.container_cpu_cfs_throttled_s...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
2    container.node-5.container_cpu_load_average_10s         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
3  container.node-5.container_fs_inodes_free./dev...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
4     container.node-5.container_fs_inodes./dev/vda1         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
5        container.node-5.container_file_descriptors        96.0       95.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                     95.0                     95.0             5.0                   0.0
6  container.node-5.container_fs_reads_merged./de...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
7       container.node-5.container_fs_reads./dev/vda         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
8      container.node-5.container_fs_reads./dev/vda1         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
9  container.node-5.container_fs_io_current./dev/...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0,                            trace_name      cmdb_id                 timestamp     value
0  trace.from_frontend-0.duration_p95  adservice-0 2022-03-20 15:26:00+08:00  0.000028
1  trace.from_frontend-2.duration_p95  adservice-0 2022-03-20 15:25:00+08:00  0.000020
2  trace.from_frontend-2.duration_p95  adservice-0 2022-03-20 15:27:00+08:00  0.000020
3  trace.from_frontend-0.duration_p95  adservice-0 2022-03-20 15:25:00+08:00  0.000018
4  trace.from_frontend-0.duration_p95  adservice-0 2022-03-20 15:24:00+08:00  0.000018
5  trace.from_frontend-1.duration_p95  adservice-0 2022-03-20 15:23:00+08:00  0.000015
6  trace.from_frontend-1.duration_p95  adservice-0 2022-03-20 15:26:00+08:00  0.000015
7  trace.from_frontend-2.duration_p95  adservice-0 2022-03-20 15:24:00+08:00  0.000014
8  trace.from_frontend-1.duration_p95  adservice-0 2022-03-20 15:25:00+08:00  0.000014
9  trace.from_frontend-2.duration_p95  adservice-0 2022-03-20 15:26:00+08:00  0.000014,        cmdb_id         log_name                 timestamp   value
0  adservice-0    log.row_count 2022-03-20 15:24:00+08:00  1560.0
1  adservice-0    log.row_count 2022-03-20 15:26:00+08:00  1526.0
2  adservice-0    log.row_count 2022-03-20 15:27:00+08:00  1468.0
3  adservice-0    log.row_count 2022-03-20 15:25:00+08:00  1456.0
4  adservice-0    log.row_count 2022-03-20 15:23:00+08:00  1348.0
5  adservice-0  log.error_count 2022-03-20 15:23:00+08:00     0.0
6  adservice-0  log.error_count 2022-03-20 15:25:00+08:00     0.0
7  adservice-0  log.error_count 2022-03-20 15:24:00+08:00     0.0
8  adservice-0  log.error_count 2022-03-20 15:26:00+08:00     0.0
9  adservice-0  log.error_count 2022-03-20 15:27:00+08:00     0.0, Empty DataFrame
Columns: [message_sample, count, earliest_timestamp, latest_timestamp]
Index: [],                                             kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window
0  container.node-5.container_cpu_cfs_throttled_p...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
1  container.node-5.container_cpu_cfs_throttled_s...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
2  container.node-5.container_fs_read_seconds./de...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
3    container.node-5.container_cpu_load_average_10s         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
4     container.node-5.container_fs_inodes./dev/vda1         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
5        container.node-5.container_file_descriptors        92.0       91.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                     92.0                     91.0
6  container.node-5.container_fs_inodes_free./dev...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
7  container.node-5.container_fs_io_time_weighted...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
8  container.node-5.container_fs_io_time_seconds....         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0
9  container.node-5.container_fs_io_current./dev/...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0,                                             kpi_name  global_p95  global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window  severity_score  breach_ratio_display
0  container.node-5.container_cpu_cfs_throttled_p...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
1  container.node-5.container_cpu_cfs_throttled_s...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
2  container.node-5.container_fs_read_seconds./de...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
3    container.node-5.container_cpu_load_average_10s         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
4     container.node-5.container_fs_inodes./dev/vda1         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
5        container.node-5.container_file_descriptors        92.0       91.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                     92.0                     91.0             5.0                   0.0
6  container.node-5.container_fs_inodes_free./dev...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
7  container.node-5.container_fs_io_time_weighted...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
8  container.node-5.container_fs_io_time_seconds....         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0
9  container.node-5.container_fs_io_current./dev/...         0.0        0.0                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                      0.0                      0.0             5.0                   0.0,                            trace_name      cmdb_id                 timestamp     value
0  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:24:00+08:00  0.000023
1  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:26:00+08:00  0.000022
2  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:25:00+08:00  0.000021
3  trace.from_frontend-2.duration_p95  adservice-1 2022-03-20 15:27:00+08:00  0.000021
4  trace.from_frontend-2.duration_p95  adservice-1 2022-03-20 15:23:00+08:00  0.000021
5  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:23:00+08:00  0.000020
6  trace.from_frontend-2.duration_p95  adservice-1 2022-03-20 15:26:00+08:00  0.000019
7  trace.from_frontend-0.duration_p95  adservice-1 2022-03-20 15:24:00+08:00  0.000019
8  trace.from_frontend-1.duration_p95  adservice-1 2022-03-20 15:27:00+08:00  0.000018
9  trace.from_frontend-2.duration_p95  adservice-1 2022-03-20 15:25:00+08:00  0.000018,        cmdb_id         log_name                 timestamp  value
0  adservice-1    log.row_count 2022-03-20 15:24:00+08:00  144.0
1  adservice-1    log.row_count 2022-03-20 15:26:00+08:00  136.0
2  adservice-1    log.row_count 2022-03-20 15:27:00+08:00  136.0
3  adservice-1    log.row_count 2022-03-20 15:25:00+08:00  126.0
4  adservice-1    log.row_count 2022-03-20 15:23:00+08:00  120.0
5  adservice-1  log.error_count 2022-03-20 15:23:00+08:00    0.0
6  adservice-1  log.error_count 2022-03-20 15:25:00+08:00    0.0
7  adservice-1  log.error_count 2022-03-20 15:24:00+08:00    0.0
8  adservice-1  log.error_count 2022-03-20 15:26:00+08:00    0.0
9  adservice-1  log.error_count 2022-03-20 15:27:00+08:00    0.0, Empty DataFrame
Columns: [message_sample, count, earliest_timestamp, latest_timestamp]
Index: [],                                             kpi_name     global_p95      global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window
0  container.node-5.container_cpu_cfs_throttled_p...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
1  container.node-5.container_cpu_cfs_throttled_s...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
2    container.node-5.container_cpu_load_average_10s       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
3  container.node-5.container_fs_io_time_seconds....       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
4     container.node-5.container_fs_inodes./dev/vda1       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
5  container.node-5.container_fs_inodes_free./dev...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
6  container.node-5.container_fs_io_time_weighted...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
7   container.node-5.container_fs_limit_MB./dev/vda1  604630.738281  604630.738281                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00            604630.738281            604630.738281
8  container.node-5.container_fs_io_current./dev/...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000
9  container.node-5.container_fs_sector_writes./d...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000,                                             kpi_name     global_p95      global_p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  run_max_value_in_window  run_min_value_in_window  severity_score  breach_ratio_display
0  container.node-5.container_cpu_cfs_throttled_p...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
1  container.node-5.container_cpu_cfs_throttled_s...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
2    container.node-5.container_cpu_load_average_10s       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
3  container.node-5.container_fs_io_time_seconds....       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
4     container.node-5.container_fs_inodes./dev/vda1       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
5  container.node-5.container_fs_inodes_free./dev...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
6  container.node-5.container_fs_io_time_weighted...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
7   container.node-5.container_fs_limit_MB./dev/vda1  604630.738281  604630.738281                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00            604630.738281            604630.738281             5.0                   0.0
8  container.node-5.container_fs_io_current./dev/...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0
9  container.node-5.container_fs_sector_writes./d...       0.000000       0.000000                       5                        5            2022-03-20 15:23:00+08:00          2022-03-20 15:27:00+08:00                 0.000000                 0.000000             5.0                   0.0,                            trace_name      cmdb_id                 timestamp     value
0  trace.from_frontend-2.duration_p95  adservice-2 2022-03-20 15:25:00+08:00  0.000026
1  trace.from_frontend-0.duration_p95  adservice-2 2022-03-20 15:27:00+08:00  0.000020
2  trace.from_frontend-0.duration_p95  adservice-2 2022-03-20 15:25:00+08:00  0.000019
3  trace.from_frontend-1.duration_p95  adservice-2 2022-03-20 15:26:00+08:00  0.000019
4  trace.from_frontend-1.duration_p95  adservice-2 2022-03-20 15:23:00+08:00  0.000017
5  trace.from_frontend-2.duration_p95  adservice-2 2022-03-20 15:23:00+08:00  0.000017
6  trace.from_frontend-2.duration_p95  adservice-2 2022-03-20 15:26:00+08:00  0.000017
7  trace.from_frontend-1.duration_p95  adservice-2 2022-03-20 15:25:00+08:00  0.000015
8  trace.from_frontend-2.duration_p95  adservice-2 2022-03-20 15:27:00+08:00  0.000015
9  trace.from_frontend-1.duration_p95  adservice-2 2022-03-20 15:27:00+08:00  0.000015,        cmdb_id         log_name                 timestamp  value
0  adservice-2    log.row_count 2022-03-20 15:24:00+08:00  148.0
1  adservice-2    log.row_count 2022-03-20 15:26:00+08:00  134.0
2  adservice-2    log.row_count 2022-03-20 15:27:00+08:00  134.0
3  adservice-2    log.row_count 2022-03-20 15:25:00+08:00  130.0
4  adservice-2    log.row_count 2022-03-20 15:23:00+08:00  110.0
5  adservice-2  log.error_count 2022-03-20 15:23:00+08:00    0.0
6  adservice-2  log.error_count 2022-03-20 15:25:00+08:00    0.0
7  adservice-2  log.error_count 2022-03-20 15:24:00+08:00    0.0
8  adservice-2  log.error_count 2022-03-20 15:26:00+08:00    0.0
9  adservice-2  log.error_count 2022-03-20 15:27:00+08:00    0.0, Empty DataFrame
Columns: [message_sample, count, earliest_timestamp, latest_timestamp]
Index: [])```
```